In [1]:
from typing import List, TypedDict

class AgentState(TypedDict):
    task: str
    plan: str
    steps: List[str]
    results: dict
    final_answer: str



In [2]:
import os
from dotenv import load_dotenv
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
# from langchain_core.tools import tool


load_dotenv()

# Initialize Tavily search tool
search_tool = TavilySearchResults(
    max_results=3,
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)
# Initialize Arxiv tool
arxiv = ArxivAPIWrapper(
    top_k_results=3,
    sort_by="relevancy",
    sort_order="descending"
)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv)


C:\Users\Uih36530\AppData\Local\Temp\ipykernel_9448\3245562815.py:12: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(


In [3]:
import operator
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.output_parsers.string import StrOutputParser

llm = ChatOllama(model="qwen2.5:7b", base_url="http://localhost:11434", temperature=0)

# 1. Plannning Node
planner_prompt = ChatPromptTemplate.from_template(
    """Create a step-by-step plan to research the following user query. 
    Your plan should be a short list of simple, actionable steps. Nothing should be there apart from list. 

    Query: {task}"""
)
planner = planner_prompt | llm | StrOutputParser()

# 2. Tool Execution Node
class ToolExecutor(BaseModel):
    """Tool execution schema."""
    tool_name: str = Field(description="The name of the tool to execute.")
    tool_input: str = Field(description="The input to pass to the tool.")

# Set up a PydanticOutputParser
parser = PydanticOutputParser(pydantic_object=ToolExecutor)

tool_executor_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", 
         '''You are an expert at choosing the correct tool and input to answer a user's question. 
         Return a JSON object that matches the following Pydantic model:

         class ToolExecutor(BaseModel):
             tool_name: str  # Must be either "tavily_search" for web search or "arxiv" for academic papers
             tool_input: str  # The search query to use
         
         Choose "arxiv" when the query is specifically about academic research, scientific papers, or technical publications.
         Choose "tavily_search" for general web searches and current information.'''),
        ("user", "Based on the following plan, what is the next tool to call and what is its input?\n\nPlan:\n{plan}\n\nCompleted Steps:\n{steps}"),
    ]
)

# tool_executor_prompt = ChatPromptTemplate.from_messages(
#     [
#         ("system", "You are an expert at choosing the correct tool and input to answer a user's question."),
#         ("user", "Based on the following plan, what is the next tool to call and what is its input?\n\nPlan:\n{plan}\n\nCompleted Steps:\n{steps}"),
#     ]
# )
tool_executor_chain = tool_executor_prompt | llm | parser

# 3. Final Answer Node
answer_prompt = ChatPromptTemplate.from_template(
    """Based on the original query and the research results, provide a comprehensive final answer.

    Query: {task}

    Research Results:
    {results}
    """
)
answer_chain = answer_prompt | llm | StrOutputParser()

In [13]:
# main.py
import os
from dotenv import load_dotenv
from typing import Literal

from langgraph.graph import StateGraph, END

from agent_state import AgentState
from tools import search_tool, arxiv_tool
from graph_nodes import planner, tool_executor_chain, answer_chain

from langfuse import Langfuse, observe

langfuse = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=os.getenv("LANGFUSE_HOST")
)

# Load environment variables
load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# --- Node Functions ---
@observe()
def plan_step(state: AgentState):
    """Generates the initial research plan."""
    print("---PLANNING---")
    plan = planner.invoke({"task": state["task"]})
    return {"plan": plan, "steps": []}

@observe()
def tool_step(state: AgentState):
    """Executes a tool based on the current plan."""
    print("---EXECUTING TOOL---")
    current_steps = state.get("steps", [])


    tool_choice = tool_executor_chain.invoke(
        {"plan": state["plan"], "steps": state["steps"]}
    )
    
    # Execute the appropriate tool based on the tool choice
    if tool_choice.tool_name == "arxiv":
        result = arxiv_tool.invoke(tool_choice.tool_input)
    else:
        result = search_tool.invoke(tool_choice.tool_input)
    
    # Update state
    new_steps = current_steps + [f"Called tool '{tool_choice.tool_name}' with input '{tool_choice.tool_input}'"]
    new_results = state.get("results", {})
    if tool_choice.tool_name not in new_results:
        new_results[tool_choice.tool_name] = []
    new_results[tool_choice.tool_name].append(result)

    return {"steps": new_steps, "results": new_results}

@observe()
def answer_step(state: AgentState):
    """Generates the final answer."""
    print("---GENERATING FINAL ANSWER---")
    final_answer = answer_chain.invoke({
        "task": state["task"],
        "results": str(state["results"])
    })
    return {"final_answer": final_answer}

# --- Conditional Edge Logic ---
@observe()
def should_continue(state: AgentState) -> Literal["continue", "end"]:
    """Determines whether to continue planning or end."""

    if len(state["steps"]) == 0:
        return "continue"
    else:
        return "end"

# --- Define the Graph ---
graph = StateGraph(AgentState)

# Add nodes
graph.add_node("planner", plan_step)
graph.add_node("tool_executor", tool_step)
graph.add_node("final_answer_generator", answer_step)

# Set the entry point
graph.set_entry_point("planner")

# Add edges
graph.add_edge("planner", "tool_executor")
graph.add_edge("final_answer_generator", END)

# Add conditional edge
graph.add_conditional_edges(
    "tool_executor",
    should_continue,
    {
        "continue": "planner",  # A more complex agent would loop back to re-plan
        "end": "final_answer_generator"
    }
)

# Compile the graph
app = graph.compile()

print(app.get_graph().draw_ascii())



      +-----------+        
      | __start__ |        
      +-----------+        
             *             
             *             
             *             
        +---------+        
        | planner |        
        +---------+        
             *             
             *             
             *             
    +---------------+      
    | tool_executor |      
    +---------------+      
             .             
             .             
             .             
+------------------------+ 
| final_answer_generator | 
+------------------------+ 
             *             
             *             
             *             
        +---------+        
        | __end__ |        
        +---------+        


In [14]:
# --- Run the Agent ---
@observe()
def run_agent(task):
    print("Starting the agent...\n")
    for event in app.stream(task):
        for key, value in event.items():
            print(f"Node '{key}' output:")
            print("---")
            print(value)
        print("\n=====================\n")

In [15]:
task = {"task": "What are the latest advancements in AI-powered drug discovery?"}
run_agent(task)

Starting the agent...

---PLANNING---
Node 'planner' output:
---
{'plan': '- Step 1: Identify key journals and publications focusing on AI in pharmaceuticals.\n- Step 2: Search for recent articles and papers using academic databases like PubMed, IEEE Xplore, or Google Scholar.\n- Step 3: Explore recent conference proceedings from major events such as NeurIPS, ICML, or the International Conference on Machine Learning in Biology and Medicine (ICMB).\n- Step 4: Review press releases and news articles from leading pharmaceutical companies and AI research institutions.\n- Step 5: Check for any relevant patents filed recently that pertain to AI in drug discovery.\n- Step 6: Look into summaries or reports from industry analysts covering the latest trends in AI-powered drug discovery.', 'steps': []}


---EXECUTING TOOL---
Node 'tool_executor' output:
---
{'steps': ["Called tool 'arxiv' with input 'AI in pharmaceuticals recent articles and papers'"], 'results': {'arxiv': ["Published: 2021-06-25